In [0]:
%pip install -U -qqqq databricks-agents mlflow langchain==0.2.16 langgraph-checkpoint==1.0.12  langchain_core langchain-community==0.2.16 langgraph==0.2.16 pydantic databricks_langchain==0.1.1 
dbutils.library.restartPython()

In [0]:
%run ./00_config

In [0]:
VECTOR_INDEX_NAME=f"{UC_CATALOG}.{UC_SCHEMA}.wikipedia_vector_index"

In [0]:
# this is cheating on our part, ideally you would have a config.yaml that you reference across notebooks

my_config = {
  "agent_prompt": """You are a highly knowledgeable and creative film assistant, specialising in analysing movie plots and helping users discover movies that match their interests or needs. You have access to a vector search retriever tool, which allows you to retrieve relevant movies based on their plot descriptions. You also have metadata on the films, such as film name, year, and category.

  Your responsibilities include:
  1. Interpreting the user's request to understand the thematic or plot-based details they are searching for.
  2. Using the retriever tool to find movies with plots that match the user's description. Retrieve the top results and summarise them clearly for the user.
  3. Providing thoughtful, concise, and contextually relevant responses to enhance the user’s understanding or interest in the films.
  4. If needed, provide additional information or insights into the films retrieved, such as key themes, genres, or cultural significance.

  When responding:
  - Always include the name of the film prominently at the start of your response, followed by a summary of its plot or other relevant details.
  - Focus on clarity and relevance to the user's query.
  - Explain why the retrieved movies fit the query when necessary.
  - Offer additional suggestions or insights to make your response engaging and helpful.

  If the user's request is ambiguous or lacks detail, ask clarifying questions before using the retriever. If the user's request is about a topic besides films, answer that you can only help with films related queries
  """,
  "temperature": 0,
  "llm_endpoint": "databricks-meta-llama-3-3-70b-instruct"
}

In [0]:
CHAT_MODEL_NAME

In [0]:
# Log the model to MLflow
import os
import mlflow
from mlflow.models.resources import (
    DatabricksVectorSearchIndex,
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
    DatabricksFunction,
    DatabricksGenieSpace,
    DatabricksTable,
)

input_example = {
    "messages": [
        {
            "role": "user",
            "content": "Find movies that address societal issues like racism or income inequality."
        }
    ]
}

with mlflow.start_run():
    logged_agent_info = mlflow.langchain.log_model(
        lc_model=os.path.join(
            os.getcwd(),
            '04_create_rag_app',
        ),
        pip_requirements=[
            "langchain==0.2.16",
            "langchain-community==0.2.16",
            "langgraph-checkpoint==1.0.12",
            "langgraph==0.2.16",
            "pydantic",
            "databricks_langchain==0.1.1", # used for the retriever tool
        ],
        model_config=my_config,
        artifact_path='agent',
        input_example=input_example,
        resources=[
        DatabricksVectorSearchIndex(index_name=VECTOR_INDEX_NAME),
        ]
    )